# Evaluating a deep-research agent

In this notebook, we'll valuate the competitor-research agent from Module 3
(`deepagents` + Tavily + Nebius Token Factory). We will:
.
- Collect golden data
- Define metrics and write up prompts for LLM judges.
- Run the pipeline under several model configurations, comparing and debugging them.
- Grade each run's saved artifacts for retrieval coverage and
  recommendation quality.
- Track cost per stage per config to visualize quality vs spend.
- Put the judge itself on trial, because a grade is only as good as the
  grader.

One warning before we start: this notebook runs the agent several times, and each run makes several dozen web searches and a number of LLM calls. Don't forget to keep track of your Token Factory and Tavily balance.

## API keys

As before, we need two:

- `NEBIUS_API_KEY` -- Nebius Token Factory (<https://tokenfactory.nebius.com>)
- `TAVILY_API_KEY` -- Tavily web search (<https://tavily.com>)

Don't forget to put an `.env` file in the folder and load the keys from it.
The cell also falls back to the bare key files `token-factory-key` and
`tavily-key` in the same folder, and fails immediately if a key is missing.

(The `.env` file is considered hidden by colab, so you won't see it unless you
specifically ask to show you hidden files.)

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

for var, key_file in (("NEBIUS_API_KEY", "token-factory-key"), ("TAVILY_API_KEY", "tavily-key")):
    if not os.environ.get(var) and os.path.exists(key_file):
        os.environ[var] = open(key_file, encoding="utf-8").read().strip()

assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in .env"
assert os.environ.get("TAVILY_API_KEY"), "Missing TAVILY_API_KEY in .env"
print("Keys loaded.")

Keys loaded.


## The agent under evaluation

The competitor-research agent is built on `deepagents` and based on the orchestrator-worker pattern. It consists of the following subagents:

- **Lead agent** (`agents.py`, `prompts.py`) -- plans the research, delegates
  to subagents, synthesises the final report. Forbidden from answering from
  its own knowledge; must gather evidence via subagents.
- **`competitor-scout`** -- discovers who the competitors are (only used when
  the initial prompt does not name them).
- **`competitor-researcher`** -- one per competitor, dispatched in parallel;
  deep-dives one company and writes a findings file.
- **`fact-checker`** -- audits the draft before it ships.

The only tools are `internet_search` (Tavily) and a virtual filesystem for
`findings/`. Every search and the URLs it returns are also written to
`tool_calls.jsonl` in the run folder; the retrieval metric reads that file.

The models used at each role are chosen at runtime via env vars
(`LEAD_MODEL`, `WORKER_MODEL`). We swap them per
configuration below to compare model choices under identical prompts and
gold data.

We'll also use a separate `JUDGE_MODEL` for LLM-as-a-judge in the evaluation pipeline.

## The evaluation harness

Everything mechanical - pricing lookup, cost tracking, `run_agent_on`, the
grading functions (`grade_retrieval`, `grade_synthesis`), and the multi-config
helpers (`set_config`, `run_deep_research`, `grade_saved_run`,
`print_config_comparison`) - lives in `eval_harness.py` so this notebook can
focus on what we're evaluating, not how. (Also it would by just too much code here otherwise.)

The `run_agent_on` function runs the agent with a given config (models +
prompt) and saves all the intermediate artifacts along with the final result
into a folder. This will allow us to evaluate all parts of the pipeline
without relaunching it.

The import below brings all of it into scope, plus three functions we use
when we put the judge itself on trial: `load_findings` and `load_report` read a saved run, and `judge_rubric` asks one judge one rubric question. Pricing is fetched at import time (needs `NEBIUS_API_KEY` in the env already).

In [2]:
!git clone -q -b evals \
    https://github.com/Nebius-Academy/ai-agents-nv.git \
    /content/ai_agents_nv

!pip install -q -r /content/ai_agents_nv/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 23.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google

In [4]:
import sys

sys.path.insert(0, "/content/ai_agents_nv")

from eval_harness import (
    set_config, run_deep_research, grade_saved_run, print_config_comparison,
    load_findings, load_report, judge_rubric, RUBRIC_PROMPT,
)

## The task

The particular task we'll be working with in this session is described in
this prompt. Two lines in it matter for the evaluation:

- *"Research in detail at least 8 competitors."* Without it the lead tends to
  pick three names up front and research only those, and then a coverage
  metric has nothing to measure.
- *"image-generation models and services, including API providers,
  creative-suite tools, and stock-photo generators"*. The market description
  steers the scout. Described only as tools for "campaign visuals and social
  ads", the scout comes back with marketing add-ons and never finds the vendors
  our gold data is about.

In [5]:
TASK_PROMPT = """\
We are preparing a launch of a new image generation service and we need to
research competitors. Research our potential competitors - image-generation
models and services, including API providers, creative-suite tools, and
stock-photo generators, that could help provide campaign visuals, social ads,
web banners, and product mockups. Research in detail at least 8 competitors.

For each of the image generation tools, compare:
  - pricing
  - commercial-use and indemnity caveats
  - workflow / API fit

Choose top-3 image generation tools and in your final report output them along with their:
  - pricing caveats (with citation, USD from the vendor's US pricing page)
  - commercial-use and indemnity caveats (with citation)
  - workflow / API fit
  - uncertainty: what could not be verified

Don't include anything else in your final report.

Every material claim -- especially about pricing, indemnity, commercial
safety, and "best" / "cheapest" -- must carry a source URL.
"""

## Golden dataset

We've put here the golden dataset we'll be working with.

First, the sources about each of the chosen image generation models: eight
competitors, each with the domains its own pages live on. A findings file that
cites `adobe.com` is evidence that the researcher looked at Adobe; a blog post
about Adobe is not. Google gets two domains because its pricing and its model
documentation live on `cloud.google.com` and `ai.google.dev`.

In [6]:
GOLD_SOURCES = {
    "Adobe Firefly": {"domains": ["adobe.com"]},
    "Canva":         {"domains": ["canva.com"]},
    "Midjourney":    {"domains": ["midjourney.com"]},
    "OpenAI":        {"domains": ["openai.com"]},
    "Stability AI":  {"domains": ["stability.ai"]},
    "Google Imagen": {"domains": ["google.com", "google.dev"]},
    "Runway":        {"domains": ["runwayml.com"]},
    "Getty Images":  {"domains": ["gettyimages.com"]},
}

And now a number of facts about those competitors: prices, plan names,
indemnity terms. Each fact names its competitor with the same key as the first
table, so the two tables can be joined. A report is not expected to state all
of them: it only analyses the competitors it picks, and we will count facts
only for those.

In [7]:
GOLD_FACTS = [
    # ---- Adobe Firefly ----
    dict(competitor="Adobe Firefly", dimension="pricing",
         claim="Adobe Firefly Standard is priced at US$9.99 per month.",
         source="https://www.adobe.com/products/firefly/plans.html"),
    dict(competitor="Adobe Firefly", dimension="pricing",
         claim="Adobe Firefly Pro is priced at US$19.99 per month.",
         source="https://www.adobe.com/products/firefly/plans.html"),
    dict(competitor="Adobe Firefly", dimension="pricing",
         claim="Adobe Firefly Premium is priced at US$199.99 per month.",
         source="https://www.adobe.com/products/firefly/plans.html"),
    dict(competitor="Adobe Firefly", dimension="commercial_safety",
         claim="Adobe describes Firefly as commercially safe.",
         source="https://business.adobe.com/products/firefly-business/firefly-ai-approach.html"),
    dict(competitor="Adobe Firefly", dimension="indemnity",
         claim="Adobe offers IP indemnification on qualifying Firefly plans.",
         source="https://business.adobe.com/products/firefly-business/firefly-ai-approach.html"),
    # ---- Canva ----
    dict(competitor="Canva", dimension="indemnity",
         claim="Canva Shield indemnifies eligible enterprise customers for AI output produced via Magic Studio.",
         source="https://www.canva.com/safe-ai-canva-shield/"),
    dict(competitor="Canva", dimension="rights_caveat",
         claim="Canva's AI product terms specify that ownership and licence rules for AI output differ from those of other Canva assets.",
         source="https://www.canva.com/policies/ai-product-terms/"),
    dict(competitor="Canva", dimension="commercial_use_caveat",
         claim="Canva's AI product terms state that the user is solely responsible for use of AI output in a commercial context and should seek professional and independent advice.",
         source="https://www.canva.com/policies/ai-product-terms/"),
    # ---- Midjourney ----
    dict(competitor="Midjourney", dimension="pricing",
         claim="Midjourney Basic plan is priced at $10 per month.",
         source="https://docs.midjourney.com/hc/en-us/articles/27870484040333-Comparing-Midjourney-Plans"),
    dict(competitor="Midjourney", dimension="pricing",
         claim="Midjourney Standard plan is priced at $30 per month.",
         source="https://docs.midjourney.com/hc/en-us/articles/27870484040333-Comparing-Midjourney-Plans"),
    dict(competitor="Midjourney", dimension="pricing",
         claim="Midjourney Pro plan is priced at $60 per month.",
         source="https://docs.midjourney.com/hc/en-us/articles/27870484040333-Comparing-Midjourney-Plans"),
    dict(competitor="Midjourney", dimension="pricing",
         claim="Midjourney Mega plan is priced at $120 per month.",
         source="https://docs.midjourney.com/hc/en-us/articles/27870484040333-Comparing-Midjourney-Plans"),
    dict(competitor="Midjourney", dimension="rights_caveat",
         claim="Midjourney's Terms of Service state that the user is responsible for all content they generate, including securing the necessary rights and permissions for any third-party intellectual property involved.",
         source="https://docs.midjourney.com/hc/en-us/articles/32083055291277-Terms-of-Service"),
    # ---- OpenAI ----
    dict(competitor="OpenAI", dimension="api_capability",
         claim="OpenAI's Image API (GPT Image models) can generate images and edit existing images from text prompts.",
         source="https://developers.openai.com/api/docs/guides/image-generation"),
    dict(competitor="OpenAI", dimension="api_pricing",
         claim="OpenAI gpt-image-2 input image tokens cost $8.00 per million tokens.",
         source="https://developers.openai.com/api/docs/pricing"),
    dict(competitor="OpenAI", dimension="api_pricing",
         claim="OpenAI gpt-image-2 output image tokens cost $30.00 per million tokens.",
         source="https://developers.openai.com/api/docs/pricing"),
    dict(competitor="OpenAI", dimension="indemnity",
         claim="OpenAI's service terms describe an indemnification for intellectual-property claims related to Output that does not apply in specific carved-out circumstances.",
         source="https://openai.com/policies/service-terms/"),
    # ---- Stability AI ----
    dict(competitor="Stability AI", dimension="pricing",
         claim="Stability AI's platform pricing uses credits at $0.01 per credit.",
         source="https://platform.stability.ai/pricing"),
    # ---- Google ----
    dict(competitor="Google Imagen", dimension="product_status",
         claim="Google Imagen models are deprecated on August 17, 2026.",
         source="https://ai.google.dev/gemini-api/docs/image-generation"),
    dict(competitor="Google Imagen", dimension="product_status",
         claim="Google's replacement for Imagen is the Nano Banana / gemini-image family.",
         source="https://ai.google.dev/gemini-api/docs/image-generation"),
    dict(competitor="Google Imagen", dimension="pricing",
         claim="Google Imagen 4 Fast costs $0.02 per image.",
         source="https://cloud.google.com/vertex-ai/generative-ai/pricing"),
    dict(competitor="Google Imagen", dimension="pricing",
         claim="Google Imagen 4 costs $0.04 per image.",
         source="https://cloud.google.com/vertex-ai/generative-ai/pricing"),
    dict(competitor="Google Imagen", dimension="pricing",
         claim="Google Imagen 4 Ultra costs $0.06 per image.",
         source="https://cloud.google.com/vertex-ai/generative-ai/pricing"),
    # ---- Runway ----
    dict(competitor="Runway", dimension="api_pricing",
         claim="Runway's API prices image generation in credits at $0.01 per credit.",
         source="https://docs.dev.runwayml.com/guides/pricing/"),
    # ---- Getty ----
    dict(competitor="Getty Images", dimension="pricing",
         claim="Getty Images' Generative AI is sold in a 25-generation pack for $49 USD.",
         source="https://www.gettyimages.com/ai"),
    dict(competitor="Getty Images", dimension="pricing",
         claim="Getty Images' Generative AI is sold in a 100-generation pack for $149 USD.",
         source="https://www.gettyimages.com/ai"),
    dict(competitor="Getty Images", dimension="commercial_safety",
         claim="Getty markets its Generative AI as trained exclusively on licensed content.",
         source="https://www.gettyimages.com/ai"),
    dict(competitor="Getty Images", dimension="indemnity",
         claim="Getty offers legal protection for its Generative AI output with cover up to $50,000 USD per image.",
         source="https://www.gettyimages.com/ai"),
]
print(f"{len(GOLD_FACTS)} gold facts about {len({f['competitor'] for f in GOLD_FACTS})} competitors")

28 gold facts about 8 competitors


# Metrics

We evaluate both the end-to-end proficiency of the pipeline and the accuracy
of its particular stages. A run logs everything into a folder: the plan, the
findings files the workers wrote, the final report, and a log of every search
with the URLs it returned. Graders use this folder, so a run can be re-graded with several judges judge without running the agent again.

## Intermediate stage check: Retrieval - did the agent search the right places?

The main metric is **`coverage`:** fraction of the 8 gold competitors whose own
domain (e.g. `adobe.com` for Adobe Firefly) is cited in a findings file the
workers wrote *and* came back from a search during the run. The second
condition is there for a reason: a worker whose searches fail can still write
a findings file out of what the model remembers from its training data, prices
and URLs included, and the URLs look right. A citation that never appeared in
any search result isn't an evidence of research, so it does not count.

**Sanity metrics** (in the same dict): `tool_calls_valid` (did the agent
make any `internet_search` at all?), `num_search_calls`, `num_findings_files`,
and how many of all the URLs cited in the findings were actually retrieved
(`citations_verified` of `citations`).

## End-to-end check - did the whole pipeline produce a good report?

**Rubric:**

* `top_3_given` - does the report present an identifiable top-3? The judge
  lists the picks it finds (`top_3_names`) and our code checks that there are
  three. Products mentioned only as alternatives or in the uncertainty
  section are not picks.
* `evidence_corresponds_to_retrieved` - for every factual claim in the
  final report, does supporting evidence appear in the pipeline's own
  `findings/*.md` files? Catches the "lead invented content the workers
  never retrieved" failure mode.
* `recommendation_grounded` - does each of the top-3 picks have its own
  findings file?

Each item comes with a `why` explanation in `rubric_reasons`, which might be useful for debugging.

All these metrics are obtained in just one judge call with structured outputs.

There are two special situations that the harness has to handle:

1. *Empty evidence*. If a run has no findings files, or its final report is empty, there is nothing to check the report against. The harness then reports the three rubric items as *not gradable* and doesn't call the judge.

  The reason for this design choice is an observation from the test runsnotebook: given a report and an empty findings section, a small judge tends to answer "every claim is supported", because the report is full of citations and the judge takes the citations for evidence. Asked to grade against nothing, it grades the formatting. So it's better not invoke the judge at all.

2. *No answer*. A judge call can also come back with no answer at all, for a reason explained in the judge section below. The harness reports that item as *no answer* rather than *NO*, and prints how many calls did this. When you read a grade, no answer means the judge never delivered a verdict, and a rerun of the grading may fill it in.

Good prompting and orchestrating may be crucial for LLM-as-a-Judge, especially if your model is
small. It's very important that your agent's prompt and your judge's prompt
are aligned. Imagine, for example, this situation:

* The agent is prompted to include only three tools in the report.
* The judge is prompted to check if the report has clearly identified top-3
  picks.

Then if the report doesn't state exactly "The top-3 picks are:..." and just
lists them, the judge may say *NO*. That is why the judge extracts the names
and the counting is left to code.

**Fact recall - one LLM call per gold fact of a competitor in scope:**

1. `competitor_in_scope` - does the report *actually analyze* this
   competitor? *YES* = it has its own section, a row in the comparison table,
   or is one of the top-3 picks. *NO* = it only appears in a passing list of
   alternatives with no per-competitor analysis. This depends on the
   competitor, not on the fact, so it is asked once per competitor and the
   answer is reused for all of its facts.
2. `contains` - a per-fact rubric that is only asked for in-scope competitors: does the report substantively state a claim equivalent to the reference fact (same numbers, same names)?

We then aggregate those per-fact answers into two numbers:

- `facts_hit` - how many gold facts of in-scope competitors got *YES*.
- `fact_recall` - `facts_hit` divided by the number of *in-scope* gold
  facts. We exclude facts about competitors the report correctly narrowed
  out, so we don't punish it for facts it wasn't supposed to cover.

Read `facts_hit` alongside `fact_recall`, and alongside the total: the cell
prints all three, so 5-out-of-5 is never mistaken for 5-out-of-28.

# The models we'll use

We'll compare models in three different roles:

* The **lead** planning model
* The **worker** model for each of the sub-agents
* The **judge** model for evaluation

Every configuration names all three. We use four models:

In [8]:
GEMMA     = "google/gemma-3-27b-it"
NANO      = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B"
LIGHTNING = "nvidia/Nemotron-3_5-Lightning"
SUPER     = "nvidia/nemotron-3-super-120b-a12b"

And we have for configs:

| config | lead | workers | judge | what it demonstrates |
|---|---|---|---|---|
| 0 | Gemma | Gemma | Nano | a pipeline that fail to launch search |
| 1 | Super | Nano | Nano | a problematic pipeline with cheap workers |
| 2 | Lightning | Lightning | Nano | the strongest pipeline of the three, but with a problematic judge |
| 3 | Lightning | Lightning | Super | config 2's run, graded by a different judge |

# Config 0. LEAD=Gemma, WORKER=Gemma, JUDGE=Nano

In [9]:
set_config(name="config0-gemma", lead=GEMMA, worker=GEMMA, judge=NANO)
run_dir_0 = run_deep_research(TASK_PROMPT)

Config 'config0-gemma':  LEAD=google/gemma-3-27b-it  |  WORKER=google/gemma-3-27b-it  |  JUDGE=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B

Run folder: runs/deep-config0-gemma-1c06ea8e
  searches: 0
  deep_research: $0.0003


Let's look at the final output:

In [10]:
from IPython.display import Markdown, display

display(Markdown(load_report(run_dir_0)))

Okay, I understand. I will act as the lead agent for a competitive research system, focusing on image generation services. I will not rely on pre-existing knowledge and will delegate research to sub-agents, synthesizing their findings into a decision-grade competitive analysis. I will adhere to the strict operating rules, especially regarding sourcing and avoiding direct answers without evidence.

Here's my plan:

**1. PLAN**

*   **SUBJECT:** New image generation service (unnamed, but implied to be the service we are launching)
*   **MARKET/CATEGORY:** Image generation – specifically for campaign visuals, social ads, web banners, and product mockups.  Buyer segment is marketing/creative professionals and businesses needing visual content.
*   **COMPETITOR SET:** Initially unknown. I will dispatch a competitor scout to identify potential competitors.  The user requested at least 8 competitors be researched in detail.
*   **DIMENSIONS:**
    *   Pricing & Packaging
    *   Commercial-Use and Indemnity Caveats
    *   Workflow / API Fit
    *   Recent Moves (last ~12 months) - *This will be useful context but not a primary output.*

**2. DISCOVER**

I will dispatch a competitor scout to enumerate the most relevant competitors.

**3. DELEGATE**

After receiving the competitor list, I will dispatch up to 8 competitor-researcher agents, one per competitor.

**4. ASSESS, 5. WRITE, 6. VERIFY, 7. ANSWER**

These steps will follow the completion of the research phase. The final output will be limited to the requested information: pricing caveats, commercial-use/indemnity caveats, workflow/API fit, and uncertainty for the top 3 image generation tools.

Now, let's start with the discovery phase.

```tool_code
from agents.tools import task

task(
    name="competitor-scout",
    description="Identify competitors in the image generation market.",
    objective="Identify at least 10 significant competitors in the image generation market, including API providers, creative-suite tools, and stock-photo generators that offer image generation capabilities. Focus on services used for campaign visuals, social ads, web banners, and product mockups.",
    instructions="""Provide a list of competitors with their homepage URLs. Prioritize services that are actively marketed towards professional content creation and businesses. Do not include simple image editing tools without generative capabilities.  Include a brief (one sentence) description of each competitor's primary focus.""",
    output_file="competitor_list.md"
)
```

Now, this is really strange. It's not a report from deep research. It's a
fragment of the planner's thinking process! But what happened?

To answer this question, let's look at the grades:

In [11]:
scores_0 = grade_saved_run(run_dir_0, GOLD_SOURCES, GOLD_FACTS)


--- Retrieval ---
  tool_calls_valid: False   num_search_calls: 0   findings files: 0   citations verified: 0/0
  coverage:         0%   (0/8)
    Adobe Firefly         MISSED
    Canva                 MISSED
    Midjourney            MISSED
    OpenAI                MISSED
    Stability AI          MISSED
    Google Imagen         MISSED
    Runway                MISSED
    Getty Images          MISSED


in-scope: 100%|██████████| 8/8 [00:08<00:00,  1.07s/it]
fact-recall: 0it [00:00, ?it/s]


--- Recommendation (judge: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B) ---
  Rubric:
    not gradable  top_3_given: no plan, notes or findings files found anywhere under the run folder
    not gradable  evidence_corresponds_to_retrieved: no plan, notes or findings files found anywhere under the run folder
    not gradable  recommendation_grounded: no plan, notes or findings files found anywhere under the run folder
  In scope: []
  Fact recall: 0 of 0 in-scope facts (28 total)   0%
  evaluation: $0.0008


We could actually stop this after the retrieval check: the agent didn't even
try to search the web.

Gemma terminated the loop on turn 1 by not requesting any tools. `deepagents`
saw "no tool calls to execute, model says stop" and treated the plan text as
the final answer. The pipeline never actually dispatched a scout, never
called a researcher, never issued a search - everything after the plan text
simply didn't happen. The rubric is *not gradable* because there are no
findings to check the report against, which is the correct thing for a
grader to say here.

# Config 1. LEAD=Super, WORKER=Nano, JUDGE=Nano

Let's now try NVIDIA models: Nemotron Super as the lead and Nemotron Nano,
the cheapest model on the list, as the researchers. (It might take some time.)

In [12]:
set_config(name="config1-lead-super-worker-nano", lead=SUPER, worker=NANO, judge=NANO)
run_dir_1 = run_deep_research(TASK_PROMPT)

Config 'config1-lead-super-worker-nano':  LEAD=nvidia/nemotron-3-super-120b-a12b  |  WORKER=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B  |  JUDGE=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B

Run folder: runs/deep-config1-lead-super-worker-nano-8daca729
  searches: 30
  deep_research: $0.4577


Let's check the final output; maybe we're luckier this time:

In [13]:
display(Markdown(load_report(run_dir_1)))



## Adobe Firefly
- **Pricing caveats**: Standard plan: $9.99 per month, 2,000 generative credits, unlimited standard image and vector generations. Pro plan: $19.99 per month, 4,000 generative credits, includes Photoshop on web/mobile and Adobe Express Premium. Pro Plus plan: $49.99 per month, 10,000 generative credits, includes admin tools and 24/7 support for teams. Premium plan: $199.99 per month, 50,000 generative credits, unlimited access to Firefly Video Model and overage options. Free tier: Limited free daily generations, no commercial use rights. [Source: https://www.adobe.com/products/firefly/plans.html (accessed 2025-11-03)]
- **Commercial-use and indemnity caveats**: Paid plans grant commercial usage rights and IP indemnification for Firefly-generated content, subject to terms of service; indemnification covers legal claims arising from usage within authorized parameters. Adobe offers copyright indemnification similar to Adobe Stock for Firefly outputs, promising to defend customers in IP infringement lawsuits, with caps that vary by plan (enterprise caps potentially $50K+). [Sources: https://business.adobe.com/products/firefly-business/firefly-ai-approach.html (accessed 2025-11-03); https://www.computerworld.com/article/1628682/adobe-offers-copyright-indemnification-for-firefly-ai-based-image-app-users.html (accessed 2025-11-03)]
- **Workflow / API fit**: Firefly is integrated across Adobe Creative Cloud apps (e.g., Photoshop, Illustrator, Express) enabling native AI-powered editing, generative fill, text‑to‑image, video, and audio generation within familiar workflows. The Firefly API allows programmatic access to image, video, and audio generation models, supporting automation and integration into external platforms, with credit‑based consumption. Credit system: usage of premium features (video, audio, partner models) consumes generative credits; each plan provides a monthly allotment of credits. Firefly includes C2PA Content Credentials for provenance metadata, enabling traceability of generated assets. [Sources: https://www.adobe.com/products/firefly.html (accessed 2025-11-03); https://community.adobe.com/questions-404/firefly-api-copyright-indemnification-and-c2pa-metadata-for-commercial-use-1549088 (accessed 2025-11-03); https://www.therundown.ai/tools/adobe-firefly (accessed 2025-11-03)]
- **Uncertainty**: Exact per-image cost in USD is not explicitly stated (credits-based model); indemnification caps vary by plan and are not fully disclosed for all tiers.

## Getty Images Generative AI
- **Pricing caveats**: $49 for 25 generations ($1.96 per image). $149 for 100 generations ($1.49 per image). Single-seat Creative or Editorial subscription pricing details are listed on Getty's live plan page. UltraPacks pricing ranges from $130 to $499 per download. [Sources: https://www.gettyimages.com/ai/generation (Date: 2025-11-03); https://photutorial.com/getty-images-pricing (Date: 2025-11-03)]
- **Commercial-use and indemnity caveats**: Generated visuals include automatic legal protection of up to $50,000 USD per image. Getty Images offers uncapped indemnification for generated content. Content is described as “commercially safe” for campaign visuals, social ads, web banners, and product mockups. [Sources: https://www.gettyimages.com/ai/generation (Date: 2025-11-03) (all three points)]
- **Workflow / API fit**: Text‑to‑image generation with options to regenerate, create variations, view generation history, and obtain high‑resolution outputs. Ability to add, remove, or replace elements in existing Getty Images library images. API integration available for enterprise customers to embed generative AI into creative applications and plugins. [Sources: https://www.gettyimages.com/ai/generation (Date: 2025-11-03) (first two points); https://newsroom.gettyimages.com/en/istock/getty-images-launches-generative-ai-by-istock-for-small-businesses-designers-and-marketers (Date: 2025-11-03) (API integration)]
- **Uncertainty**: Detailed indemnity terms beyond the $50,000 per‑image protection are not publicly disclosed. Specific enterprise contract pricing and custom allowance details are not publicly available.

## OpenAI DALL·E 3 (API)
- **Pricing caveats**: DALL·E 3 – Standard: 1024 × 1024: $0.040 per image; 1024 × 1792 / 1792 × 1024: $0.080 per image. DALL·E 3 – HD: 1024 × 1024: $0.080 per image; 1024 × 1792 / 1792 × 1024: $0.120 per image. DALL·E 2: 1024 × 1024: $0.020 per image; 512 × 512: $0.018 per image; 256 × 256: $0.016 per image. Pricing is per generated image, billed per API call; bulk usage may qualify for volume discounts or custom enterprise pricing. [Source: OpenAI API docs – “Vision API (/v1/images/generations – dall-e-3) – token usage information” (2023) (all DALL·E 3 rates); OpenAI API docs (legacy) for DALL·E 2 rates]
- **Commercial-use and indemnity caveats**: Commercial Use Permission: Users may use DALL·E outputs for “any legal purpose, including commercial use” (e.g., prints, merchandise, advertising, stock photos). Ownership of Output: All right, title, and interest in DALL·E outputs are assigned to the user. Indemnification: OpenAI indemnifies API customers (including enterprise) against third‑party IP claims arising from the output, provided the user did not knowingly infringe, did not disable safety filters, and complied with documentation. Limitations of Indemnity: No coverage if: (i) the user knew or should have known the output was infringing; (ii) safety filters were disabled; (iii) the output was modified or combined with non‑OpenAI products in a way that creates infringement. Beta / Preview Services: Indemnity does not apply to beta/preview services; they are provided “as‑is”. Copyright Shield: OpenAI has announced a “Copyright Shield” for enterprise customers that will cover legal costs if the output is sued for infringement, but this is not yet active for general API users. [Sources: Terms.Law AI‑Output Rights summary (2026) (Commercial Use & Ownership); OpenAI Service Terms (2023) (Indemnification & Limitations); OpenAI Service Terms – Beta Services (Beta/Preview); Community discussion (2025) (Copyright Shield)]
- **Workflow / API fit**: API Endpoints: Vision API (/v1/images/generations) is the primary endpoint for DALL·E 3 image generation. Supports `n` parameter to generate multiple images per request (charged per image). Supports `size` parameter limited to 256x256, 512x512, 1024x1024 (for DALL·E 2) and similar options for DALL·E 3 (standard/HD). Rate Limits: Tiered requests‑per‑minute (RPM) limits: Tier 1 – 500 img/min; Tier 2 – 2,500 img/min; Tier 3 – 5,000 img/min; Tier 4 – 7,500 img/min; Tier 5 – 10,000 img/min. Higher tiers are unlocked with increased spend. Batch & Cost Savings: Batching multiple prompts in a single request can reduce per‑image cost (OpenAI mentions “half‑off” for batching). Third‑party aggregators (e.g., GetGoAPI) claim up to 20 % savings but require foreign payment methods and may need VPN. Integration Points: Can be called from any environment that can make HTTPS requests (Python, Node.js, etc.). Returns image URLs directly; no need for separate storage unless desired. Supports both standard and HD quality outputs. Safety & Filtering: Built‑in content filters (e.g., to block depictions of real people, living artists, or copyrighted styles). Users can disable or adjust filters, but doing so may affect indemnity coverage. Typical Workflow for Campaign Visuals: 1. Prepare prompt with brand‑specific guidance. 2. Choose size/quality (standard vs. HD). 3. Call /v1/images/generations with desired parameters (e.g., `n=4`, `size=1024x1792`). 4. Retrieve image URLs, download, and use in assets. 5. Ensure safety filters are enabled to avoid indemnity exclusions. [Source: findings/openai_dalle3.md lines 49-55 (all workflow points)]
- **Uncertainty**: Official OpenAI Pricing Page: No direct, publicly accessible pricing page from OpenAI for DALL·E 3 API; rates are documented only in developer community posts and API reference sites. Enterprise‑Specific Discounts: Exact volume‑discount tiers beyond the generic “Tier 1‑5” description are not publicly disclosed. Copyright Shield Availability: The announced indemnity shield for broader API users has not yet been released; details on eligibility and activation timing are unclear. Azure OpenAI Service Pricing: No explicit per‑image rates are published; enterprise customers must request a quote. Third‑Party Cost Savings Claims: Independent platforms claim lower pricing, but their legitimacy, payment‑method requirements, and compliance with OpenAI’s terms are not verified.

At least the report should make sense: three products, each with the four
sections the task asked for, every claim with a URL and a date. Let's look at
the metrics.

In [14]:
scores_1 = grade_saved_run(run_dir_1, GOLD_SOURCES, GOLD_FACTS)


--- Retrieval ---
  tool_calls_valid: True   num_search_calls: 30   findings files: 8   citations verified: 51/56
  coverage:         50%   (4/8)
    Adobe Firefly         found
    Canva                 found
    Midjourney            MISSED
    OpenAI                MISSED
    Stability AI          found
    Google Imagen         MISSED
    Runway                MISSED
    Getty Images          found


fact-recall: 100%|██████████| 13/13 [00:39<00:00,  3.04s/it]


--- Recommendation (judge: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B) ---
  Rubric:
    YES  top_3_given: picks: ['Adobe Firefly', 'Getty Images Generative AI', 'OpenAI DALL·E 3 (API)']. The final report presents thr
    YES  evidence_corresponds_to_retrieved: All factual claims in the report are directly supported by citations that appear in the retrieved findings for
    YES  recommendation_grounded: Each of the three named competitor tools (Adobe Firefly, Getty Images Generative AI, OpenAI DALL·E 3) has a de
  In scope: ['Adobe Firefly', 'Getty Images', 'OpenAI']
  Fact recall: 9 of 13 in-scope facts (28 total)   69%
  evaluation: $0.0086


In your experiment, you might get different results, but chances are the
metrics look reasonable: some coverage, a rubric that mostly passes.

But let's perform a more thorough audit.

In [20]:
import json
from pathlib import Path

def show_run(run_dir):
    """What the run folder says about how the research was done."""
    run_dir = Path(run_dir)

    # 1. files, with the folder they landed in
    print("findings files:")
    findings = sorted(run_dir.rglob("findings/*.md"))
    for f in findings:
        print(f"  {f.stat().st_size:6d} B  {f.relative_to(run_dir)}")
    for name in ("plan.md", "notes.md"):
        hits = list(run_dir.rglob(name))
        print(f"  {name}: {hits[0].relative_to(run_dir) if hits else 'MISSING'}")

    # 2. what the lead did, from its own conversation
    msgs = json.loads((run_dir / "messages.json").read_text(encoding="utf-8"))
    dispatched = Counter({"competitor-scout": 0, "competitor-researcher": 0, "fact-checker": 0})
    per_target = Counter()      # first words of each researcher task -> calls
    task_ids = set()
    lead_searches = 0
    for m in msgs:
        for tc in m.get("tool_calls") or []:
            if tc["name"] == "task":
                kind = tc["args"].get("subagent_type", "?")
                dispatched[kind] += 1
                task_ids.add(tc["id"])
                if kind == "competitor-researcher":
                    per_target[" ".join(tc["args"].get("description", "").split()[:3])] += 1
            elif tc["name"] == "internet_search":
                lead_searches += 1
    empty = sum(1 for m in msgs if m.get("type") == "tool"
                and m.get("tool_call_id") in task_ids and not str(m.get("content", "")).strip())
    print("\nlead's dispatches:", dict(dispatched))
    print(f"  task results that came back empty: {empty}")
    redispatched = {k: v for k, v in per_target.items() if v > 1}
    if redispatched:
        print(f"  competitors dispatched more than once: {redispatched}")
    print(f"  searches made by the lead itself: {lead_searches}")

    # 3. how much searching the workers did
    trace = run_dir / "tool_calls.jsonl"
    calls = [json.loads(l) for l in trace.read_text(encoding="utf-8").splitlines() if l.strip()]
    queries = [c["args"]["query"] for c in calls if c["tool"] == "internet_search"]
    n_workers = max(1, len([f for f in findings if "scout" not in f.name.lower()]))
    print(f"\n{len(queries)} searches, about {len(queries) / n_workers:.1f} per findings file")
    for q, n in Counter(queries).most_common(3):
        if n > 1:
            print(f"  {n}x  {q}")

    # 4. the dates the workers wrote next to their sources
    dates = Counter()
    for f in findings:
        dates.update(re.findall(r"20\d\d-\d\d-\d\d", f.read_text(encoding="utf-8")))
    print(f"\ntoday is {date.today()}; most common dates next to sources: {dict(dates.most_common(3))}")


show_run(run_dir_1)

findings files:
    3633 B  findings/adobe_firefly.md
    3074 B  findings/blackforest_fluxpro.md
    2263 B  findings/canva_magicmedia.md
    2249 B  findings/getty_genai.md
    7687 B  findings/openai_dalle3.md
    1347 B  findings/scout.md
    2569 B  findings/stability_sd3.md
    4130 B  mnt/data/findings/ideogram.md
      11 B  mnt/data/findings/scout.md
    4376 B  mnt/data/findings/shutterstock_genai.md
  plan.md: mnt/data/plan.md
  notes.md: MISSING

lead's dispatches: {'competitor-scout': 2, 'competitor-researcher': 9, 'fact-checker': 0}
  task results that came back empty: 2
  competitors dispatched more than once: {'Research competitor: OpenAI': 2}
  searches made by the lead itself: 1

30 searches, about 3.8 per findings file
  2x  OpenAI DALL·E 3 API pricing

today is 2026-09-09; most common dates next to sources: {'2025-11-03': 37, '2025-09-29': 13, '2025-09-28': 11}


In [21]:
def recall_by_dimension(scores):
    """Fact recall split by the kind of fact, from grade_saved_run's output."""
    hit, total = Counter(), Counter()
    for pf in scores["recommendation"]["per_fact"]:
        total[pf["dimension"]] += 1
        hit[pf["dimension"]] += pf["contains"]
    print(f"{'dimension':22s} hit / in scope")
    for dim in sorted(total):
        print(f"{dim:22s} {hit[dim]:3d} / {total[dim]}")


recall_by_dimension(scores_1)

dimension              hit / in scope
api_capability           0 / 1
api_pricing              0 / 2
commercial_safety        1 / 2
indemnity                3 / 3
pricing                  5 / 5


What you are likely to see here, and what it means:

* **Dispatch lines**. The lead usually sends out one researcher per competitor and gets a few empty replies back, so some competitors are dispatched twice; it often searches the web itself to fill a gap, which its prompt forbids; and it does not call the fact-checker. `notes.md` is usually missing.

  So the one step in the pipeline that was designed to catch a thin or wrong findings file typically doesn't run, and the lead covers for its workers instead of checking them.

* **The files**. The workers write short files from a few searches each, and they may land in more than one folder, because the agent's prompts don't prescribe the paths.

  The latter might become a problem during debugging: your scripts may fail to find the findings files, in which case they'll report that no files were written. And indeed, our evaluation harness has to do recursive search for the findings files to account for it.

  Short files isn't bad by itself: a worker that finds the vendor's pricing and terms pages can cover the task in a few kilobytes. However, too many short files mean that the workers don't do their job too well.

* **The dates**. Compare the most common dates with today's date. If they are months or a year apart, it is likely that the worker didn't record the date when it fetched the page; it wrote a date it had in mind.

Let's see if a different choice of LLMs makes things better.

# Config 2. LEAD=Lightning, WORKER=Lightning, JUDGE=Nano

Note that the lead is now more lightweight, while the workers will be more capable, even if having around the same price as Nano.

In [26]:
set_config(name="config2-lightning", lead=LIGHTNING, worker=LIGHTNING, judge=NANO)
run_dir_2 = run_deep_research(TASK_PROMPT)

Config 'config2-lightning':  LEAD=nvidia/Nemotron-3_5-Lightning  |  WORKER=nvidia/Nemotron-3_5-Lightning  |  JUDGE=nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B

Run folder: runs/deep-config2-lightning-78a28307
  searches: 72
  deep_research: $0.6439


Let's look at the final output:

In [27]:
display(Markdown(load_report(run_dir_2)))


<tool_call>
<function=write_file>
<parameter=content>
# Competitive Analysis: Image Generation Services for Campaign Visuals

## Comparison Matrix (8 Competitors)

| Competitor | Pricing (USD, US page) | Commercial-Use & Indemnity | Workflow / API Fit |
|------------|------------------------|----------------------------|---------------------|
| **Adobe Firefly** | Standard $9.99/mo (2K credits), Pro $19.99/mo (4K), Pro Plus $49.99/mo (10K), Premium $199.99/mo (50K). All paid plans include unlimited standard image generations. [adobe.com/products/firefly/plans.html] | Paid plans include IP indemnification (Creative Cloud). Free tier: limited commercial use, no indemnity. Enterprise ETLA: full indemnity. Caveats: covers only training-data claims; does NOT cover user-input content; caps/carve-outs; risky prompts (e.g., copyrighted characters) excluded. [findings/adobe-firefly.md] | Firefly Services API: Text-to-Image, Generative Fill, Generative Expand, Composite Operations, Custom Models API. Integrates with Adobe GenStudio, Express, Photoshop. API requires Enterprise contract for Custom Models & full indemnity. Supports brand-consistent generation at scale. [findings/adobe-firefly.md] |
| **OpenAI GPT Image API** | GPT Image 2: Low $0.006/1024x1024, Medium $0.053, High $0.211. GPT Image 1.5: Low $0.009, Medium $0.034, High $0.133. GPT Image 1: Low $0.011, Medium $0.042, High $0.167. GPT Image 1 Mini: $0.005–$0.052/image. $5 free credits new users. [developers.openai.com] | Ownership assigned to user. Standard indemnification: OpenAI defends against third-party IP infringement claims for Output, except if customer knew/should have known infringing or disabled safety features. "Copyright Shield" (DevDay June 2025) expands coverage to API users (initially Enterprise). Liability cap: $100 aggregate. [findings/openai-gpt-image.md] | Primary endpoints: Images API (POST /v1/images/generations, /edits), Responses API with image_generation tool. Models: GPT Image 2.5 recommended, 1/1.5/2 tiers. Batch processing, async support. Image sizes: 1024x1024, 1024x1536, 1536x1024. Suitable for programmatic pipelines. [findings/openai-gpt-image.md] |
| **Google Imagen Vertex AI** | Imagen 4 Fast: $0.02/1024x1024. Imagen 4 Standard: $0.04/1024x1024. Imagen 4 Ultra: $0.06/1024x1024. Imagen 3 Fast/Standard same rates. Imagen 2 (legacy): $0.02/1024x1024. Upscaling $0.003/image. [intuitionlabs.ai citing Google Vertex AI] | Commercial use permitted across enterprise tiers (Vertex AI, Gemini Advanced/Business/Enterprise). Two-pronged indemnity: (1) training-data indemnity, (2) generated-output indemnity covering third-party IP claims including copyright. Applies if user follows responsible AI practices. Does NOT apply if deliberately used to create infringing content. Enterprise Vertex AI customers get strongest IP indemnification. [findings/google-imagen.md] | RESTful API via Google Cloud Vertex AI publisher endpoint. Accessible through Model Garden. Integrates via Workflows connector using generateContent method (supports multimodal inputs). Model selection: imagen-3.0-generate-002, imagen-4.0-ultra-generate-001, etc. Supports upscaling. [findings/google-imagen.md] |
| **Shutterstock AI** | AI Generative Plus: $29/mo (monthly) or $180/yr ($15/mo effective). AI Plus (entry): $7/mo for 100 credits/mo (4 variations each = 400 images). Higher tiers: Professional ~$199/mo yearly, Team $489/mo. Per-image credits: 48¢–$2.90 depending on package. Standard license includes 50 AI generation credits/mo. [shutterstock.com/pricing] | Standard License: AI-generated images usable commercially (ads, marketing, packaging) under Standard or Enhanced license. Enhanced adds: unlimited reproduction, merchandise resale, unlimited digital impressions, trademark/logo use. Key caveat: images must meet content standards & be approved by review team. Enterprise-only full indemnification (legal + financial protection). Standard license provides some indemnification (~$10K). Human review required for full indemnification. [findings/shutterstock-ai.md] | REST API: 109 endpoints for search, keyword extraction, account/media management. AI generation via web UI only — NOT publicly documented as callable REST API for automated generation. API focuses on library search/licensing of AI-generated content once created in UI. Creative Flow toolkit integrates AI generation alongside stock search/editing. ChatGPT app integration available. 3D API (text-to-3D) released July 2024. [findings/shutterstock-ai.md] |
| **Midjourney** | Basic: $10/mo / $8/mo annual. Standard: $30/mo / $24/mo annual. Pro: $60/mo / $48/mo annual. Mega: $120/mo / $96/mo annual. Additional Fast GPU hours: $4/hr (any time). No free tier (removed March 2023). [midjourney.com pricing] | Any paid subscription (Basic–Mega) grants commercial usage rights. Revenue threshold: companies with >$1M annual gross revenue MUST upgrade to Pro or Mega for "company ownership" classification. Free trial users: NO commercial rights (CC BY-NC 4.0). NO copyright indemnification: outputs "as is"; users bear all IP risk; must indemnify and hold harmless Midjourney. [findings/midjourney.md] | API is async/task-based: submit imagine/blend/change → receive task ID → poll /mj/task/{task_id}/fetch until SUCCESS/FAILURE → retrieve image URLs. Key endpoints: /mj/submit/imagine, /mj/task/{task_id}/fetch. Actions: upscale, variation, zoom, pan, remix. Auth: API key/bearer token. Rate limits per plan. Suitable for programmatic pipelines (submit, wait, download), not low-latency interactive use. [findings/midjourney.md] |
| **Canva Magic Media** | Free: $0/mo, up to 200 Standard AI uses/mo. Pro: $180/yr per person, up to 200 Standard or 20 Premium AI uses/mo. Business: $250/yr per person, up to 200 Standard or 20 Premium AI uses/mo. AI Pass add-on: $100/person/mo (multiplies allowance ~40× Pro, 20× Business). Magic Media included within AI allowance, no separate price. [canva.com/en/pricing] | Pro plan: all content (including AI-generated) can be used commercially, no attribution required. Critical restriction: AI content should not be used to create content mistaken for photographs of real people. Free plan: limited commercial rights; premium elements watermarked; commercial use not permitted without upgrading. Business: same commercial rights as Pro + team admin controls. Zero IP indemnification for Free/Pro; users bear full legal exposure; Canva's max liability = greater of $100 USD or 12-month subscription fees. Enterprise (100+ seats): only tier with IP indemnification. [findings/canva-magic-media.md] | Native Magic Media has NO public API endpoint; cannot be triggered programmatically from external workflows. Workaround: generate via external AI API, then upload to Canva via Connect API / Asset Upload. Canva Apps SDK & Connect APIs restricted to paid plans. Zapier integration exists for general Canva actions but no "generate image via Magic Media" step. Programmatic workflow requires external AI API + Canva upload. [findings/canva-magic-media.md] |
| **Claid.ai** | Free trial: 50 credits + 50 API credits. Web Essentials: $9/mo yearly ($15/mo monthly) = 500 credits/mo. Pro: $35/mo yearly ($49/mo monthly) = 2,000 credits/mo. Business: custom-priced. API: $59 one-time for 1,000 credits. Credit consumption: Upscale 2-8 credits, Remove background 1 credit, AI Photoshoot 4 credits max, AI Video 60 credits/5s, Outpaint 2 credits, AI Edit 4 credits. [claid.ai pricing] | No explicit commercial-use prohibition, but user bears full responsibility ensuring outputs do not infringe third-party IP. No platform-wide commercial-use license or indemnity granted — commercial use permitted only as far as underlying rights allow; user assumes all IP risk. Terms impose general "do not infringe proprietary rights" obligation. No copyright/IP indemnification for users; user-to-provider indemnity only (user defends Claid). [findings/claid-ai.md] | API architecture: Image Editing API, Image Generation API, AI Photoshoot API. Operations chained within same area in one request; cross-area combos need two requests. Batch-ready for catalog/workflow processing. Async jobs with webhooks for high-volume processing. Default rate limits: 4 RPS / 120 RPM; custom limits available. Strongest for marketplace/ecommerce image workflows (printing, real estate, food delivery, automotive). [findings/claid-ai.md] |
| **Bria** | Free tier: 100 free generations. Pay-as-you-go: starting from $0.018 per generation. Full price list: Fibo $0.03/Image, Fibo Lite $0.02/Image, Fibo Structured Prompt $0.02/Call. Enterprise tier: custom pricing, unlimited IP indemnification. [bria.ai/pricing] | Free & Pay-as-you-go tiers: "Capped standard indemnification." Enterprise tier: "Unlimited IP indemnification." Models trained exclusively on 100% licensed, safe-for-commercial-use data. Patent attribution engine traces output back to training data sources, compensates content partners. Indemnity covers copyright infringement but may not cover all IP or privacy claims. Full indemnity against copyright infringement for all outputs, enterprise customers only. [findings/bria.md] | API workflow (3-step sync): 1. Submit request (sync=false default, response includes status_url, request_id), 2. Poll status_url or /status/{request_id} until completed, 3. Retrieve output. Capabilities: generate images from text prompts, train tailored models, generate ads at scale via Ads Generation APIs, control via Visual Generative Language (VGL): structured JSON for lighting, camera, composition, objects, style, season, etc. VGL integrates into code pipelines, CI/CD, version control. [findings/bria.md] |

---

## Top-3 Image Generation Tools

### 1. Adobe Firefly

**Pricing caveats (USD, US pricing page):**
- Standard: $9.99/mo for 2,000 generative credits (unlimited standard image generations included).
- Pro: $19.99/mo for 4,000 credits.
- Pro Plus: $49.99/mo (regularly) for 10,000 credits.
- Premium: $199.99/mo (regularly) for 50,000 credits, includes unlimited Adobe Firefly Video Model access.
- All paid plans include unlimited standard image and vector generations; generative credits used for premium features (video, audio, partner models).
- Source: https://www.adobe.com/products/firefly/plans.html (accessed August 2026) [findings/adobe-firefly.md]

**Commercial-use and indemnity caveats:**
- Platform designed "safe for commercial use"; training data consists exclusively of licensed Adobe Stock, public domain, and openly licensed content (no scraped web images).
- Free tier: limited commercial use; no IP indemnification.
- All paid Creative Cloud plans include IP indemnification: Adobe will defend customers if someone claims Firefly output infringes their IP.
- Enterprise ETLA: full enterprise indemnification via negotiated terms.
- Key caveats: indemnification covers only claims arising from Adobe's training data; does NOT cover content the enterprise itself uses as input (e.g., uploading reference material without rights); has caps and carve-outs; does not apply to obviously risky prompt use (e.g., adding copyrighted characters like Spiderman); purely AI-generated elements have limited copyright protection in the US; enterprise customers should retain generations and prompt text for record-keeping.
- Source: Multiple sources including terms.law, bestnegotiationconsultingfirms.com, content.shi.com PDF [findings/adobe-firefly.md]

**Workflow / API fit:**
- Firefly Services API capabilities: Text-to-Image, Generative Fill, Generative Expand, Composite Operations (Object/Adaptive/Precise Composite), Custom Models API for brand-tailored generation.
- Integrates with Adobe GenStudio, Adobe Express, Photoshop.
- API requires Enterprise contract for Custom Models, composite operations, and brand-consistent generation at scale.
- Supports campaign visuals via web app, Express integration; social ads via Express text-to-image; web banners via Generative Fill/Expand; product mockups via Firefly AI Assistant (beta) and Photoshop integration.
- Source: developer.adobe.com/firefly-services/docs/firefly-api [findings/adobe-firefly.md]

**Uncertainty (could not verify):**
- Exact indemnification caps/negotiated terms for Enterprise ETLA (not publicly documented; requires Adobe account representative).
- Whether Firefly API's "unlimited standard image generations" on paid plans truly has no per-image cost beyond the flat monthly fee, or if hidden usage thresholds apply.
- Real-world performance of Custom Models API for brand consistency at scale (documented only in beta/early-access contexts).
- Promotional pricing vs. listed regular prices on the US pricing page.

---

### 2. OpenAI GPT Image API

**Pricing caveats (USD, from OpenAI developer docs / US pricing page):**
- GPT Image 2 (flagship): Low quality $0.006/1024x1024, Medium $0.053/1024x1024, High $0.211/1024x1024. Token rates: $8.00/M input, $2.00/M cached input, $30.00/M output (standard); batch rates $4.00/M input, $1.00/M cached, $15.00/M output.
- GPT Image 1.5: Low $0.009, Medium $0.034, High $0.133/1024x1024.
- GPT Image 1 (deprecating Oct 2026): Low $0.011, Medium $0.042, High $0.167/1024x1024.
- GPT Image 1 Mini (cheapest): $0.005–$0.052 per image depending on quality/tier; ~80–90% cheaper than flagship High tier.
- $5 free credits for new users (no credit card required).
- Source: https://developers.openai.com/api/docs/guides/image-generation and https://developers.openai.com/api/docs/pricing (Aug 2026) [findings/openai-gpt-image.md]

**Commercial-use and indemnity caveats:**
- Ownership of outputs assigned to the user per OpenAI's service terms.
- Standard indemnification: OpenAI indemnifies API customers against third-party IP infringement claims for Output, but does NOT apply where: (i) customer or end users knew/should have known the Output was infringing/likely infringing, (ii) customer disabled/ignored safety features/filters.
- "Copyright Shield" announced at DevDay (June 2025): OpenAI will defend API customers and pay costs incurred if facing legal claims around copyright infringement. Initially for Enterprise, expanding to API users.
- Liability cap: $100 per ToS; aggregate liability under ToS is greater of amount paid for service during 12 months before claim or $100.
- For non-Enterprise API users: standard indemnity applies with the noted caveats (knowledge of infringement, failure to use safety features). Copyright Shield coverage scope for API users beyond Enterprise is not fully clarified.
- Source: https://openai.com/policies/service-

In [29]:
scores_2 = grade_saved_run(run_dir_2, GOLD_SOURCES, GOLD_FACTS)
show_run(run_dir_2)


--- Retrieval ---
  tool_calls_valid: True   num_search_calls: 72   findings files: 8   citations verified: 66/78
  coverage:         62%   (5/8)
    Adobe Firefly         found
    Canva                 found
    Midjourney            found
    OpenAI                found
    Stability AI          MISSED
    Google Imagen         found
    Runway                MISSED
    Getty Images          MISSED


fact-recall: 100%|██████████| 22/22 [01:34<00:00,  4.29s/it]


--- Recommendation (judge: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B) ---
  Rubric:
     NO  top_3_given: picks: []. The report only identifies two top picks (Adobe Firefly and OpenAI GPT Image API) and does not prov
    YES  evidence_corresponds_to_retrieved: All factual claims in the final report are supported by the retrieved findings; each competitor's details are 
     NO  recommendation_grounded: The report fails to present three recommended picks; it only lists two, so the requirement for a top-3 list is
  In scope: ['Adobe Firefly', 'Canva', 'Google Imagen', 'Midjourney', 'OpenAI']
  Fact recall: 16 of 22 in-scope facts (28 total)   73%
  judge gave no answer on 1 call(s)
  evaluation: $0.0169
findings files:
    9527 B  findings/adobe-firefly.md
    9376 B  findings/bria.md
    6846 B  findings/canva-magic-media.md
   10005 B  findings/claid-ai.md
    6801 B  findings/google-imagen.md
    3450 B  findings/midjourney.md
    6276 B  findings/openai-gpt-image.md
    4100 B  findings

Compare this with config 1: the run is faster, because Lightning tends to
dispatch all its researchers in one turn instead of one after another; the
findings files are longer; queries are not repeated; workers do not come back
empty.

It is still unlikely to be a clean run. The lead will skip the plan
file, the notes and the fact-checker, but that's not a great deal of trouble. What's worse:

* The lead ignores the "don't include anything else" prompt while writing up the final answer, including an executive summary or a comparison matrix the task never asked for.

  Because of this, the judge is likely to get confused, giving a *NO* to both `top_3_given` and `recommendation_grounded`, even though the report should be actually quite good.

  This is one of the dangers of LLM judges, especially smaller ones: they are too rigid in scoring and fail when the formatting's not right.

Sometimes, you can observe the judge giving no answer. A possible reason is the fact that Nano is a *reasoning model*: before answering, it writes a hidden
chain of thought, which counts against the output budget. On a prompt this
long — all the findings files plus the report, tens of thousands of tokens —
Nano's reasoning can run until the budget is exhausted, and the reply arrives
with no answer in it.

So, let's try a larger judge!

## Config 3. LEAD=Lightning, WORKER=Lightning, JUDGE=Super

Config 3 changes the judge and nothing else, so we don't need to run the agent itself.

In [30]:
set_config(name="config3-lightning-super-judge", lead=LIGHTNING, worker=LIGHTNING, judge=SUPER)
scores_3 = grade_saved_run(run_dir_2, GOLD_SOURCES, GOLD_FACTS)

Config 'config3-lightning-super-judge':  LEAD=nvidia/Nemotron-3_5-Lightning  |  WORKER=nvidia/Nemotron-3_5-Lightning  |  JUDGE=nvidia/nemotron-3-super-120b-a12b

--- Retrieval ---
  tool_calls_valid: True   num_search_calls: 72   findings files: 8   citations verified: 66/78
  coverage:         62%   (5/8)
    Adobe Firefly         found
    Canva                 found
    Midjourney            found
    OpenAI                found
    Stability AI          MISSED
    Google Imagen         found
    Runway                MISSED
    Getty Images          MISSED


fact-recall: 100%|██████████| 22/22 [00:50<00:00,  2.28s/it]


--- Recommendation (judge: nvidia/nemotron-3-super-120b-a12b) ---
  Rubric:
    YES  top_3_given: picks: ['Adobe Firefly', 'OpenAI GPT Image API', 'Google Imagen Vertex AI']. The report selects these three ba
    YES  evidence_corresponds_to_retrieved: Every factual claim in the report (e.g., pricing details, indemnity descriptions, API capabilities) is directl
    YES  recommendation_grounded: Each of the three recommended picks (Adobe Firefly, OpenAI GPT Image API, Google Imagen Vertex AI) has its own
  In scope: ['Adobe Firefly', 'Canva', 'Google Imagen', 'Midjourney', 'OpenAI']
  Fact recall: 17 of 22 in-scope facts (28 total)   77%
  evaluation: $0.0593


As you see, the larger model scores both `top_3_given` and `recommendation_grounded` correctly.

# Cost comparison

Let's compare quality against cost.

As you see, the best setup is the most expensive, but the difference isn't orders of magnitude.

In [32]:
from eval_harness import STAGE_COST

STAGE_COST[("config3-lightning-super-judge", "deep_research")] = \
    STAGE_COST[("config2-lightning", "deep_research")]

print_config_comparison()

  stage                                            config0-gemma    config1-lead-super-worker-nano                 config2-lightning     config3-lightning-super-judge
  ----------------------------  --------------------------------  --------------------------------  --------------------------------  --------------------------------
  deep_research                                          $0.0003                           $0.4577                           $0.6439                           $0.6439
  evaluation                                             $0.0008                           $0.0086                           $0.0169                           $0.0593
  ----------------------------  --------------------------------  --------------------------------  --------------------------------  --------------------------------
  TOTAL                                                  $0.0012                           $0.4663                           $0.6608                           $0.703

The pipeline runs are dominated by search costs, not model tokens. Grading is
cheap in comparison even with Super.

### Practice

**A planted mistake**

This is yet another demonstration of the difference between Nano and Super - this time on a report which is known to have an error.

Take Config 2's report and change one number - say, change the price of Adobe Firefly's Standard plan from $9.99 to $29.99. Don't change it in the findings.

Ask each of the three models - Nano, Lightning, and Super - the corresponding rubric question twice: once about the report as written, once about the altered copy. A judge that compares report with findings should say *evidence ok* for the first and *unsupported* for the second. A judge that grades the look of the report will say *evidence ok* to both, because the altered report is just as well cited as the original.

LLMs are stochastic, so the result may change from run to run. Try this `N_JUDGE_REPEATS` (say, 3, 5, or 10) times.

Compare the verdicts of Nano, Lightning, and Super.